# Sanity check: feature extraction pipeline

End-to-end test of the transfer learning experiment pipeline on a single (model, dataset) combination before scaling to all 60.

**What this notebook does:**
1. Verify GPU access
2. Mount Drive (where checkpoints live)
3. Clone the repo from GitHub
4. Load one model (learned_seed42)
5. Download DTD dataset (smallest of the 5)
6. Extract features on train + test splits
7. Verify feature shape and sanity-check with kNN baseline

If everything works, scale up via `02_main_workflow.ipynb`.

---

## 🔧 Configuration required

Before running, you must set **one path** in this notebook (Cell 5):

- `CHECKPOINT_ROOT` → folder on your Google Drive that holds the 12 pretrained checkpoints. The folder should contain subfolders named `{pe_type}_seed{seed}/best_model.pth` (e.g. `learned_seed42/best_model.pth`).

Download the pretrained checkpoints from the link in the repository README and place them in a folder under your Drive. Then update `CHECKPOINT_ROOT` accordingly.

All other paths use ephemeral `/content/` storage and do **not** need to be changed.

In [ ]:
# 1. GPU check
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# 2. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Clone repo from GitHub  (no changes needed)
import shutil, os
REPO_DST = '/content/vit-pe-transfer'
if os.path.exists(REPO_DST):
    shutil.rmtree(REPO_DST)
!git clone https://github.com/djokobandjur/vit-pe-transfer.git $REPO_DST
%cd $REPO_DST

In [ ]:
# 4. Add repo to Python path and verify imports  (no changes needed)
import sys
sys.path.insert(0, REPO_DST)

from models import load_pretrained_model, PE_TYPES, SEEDS, list_available_checkpoints
from data import get_dataloader, DATASETS, DATASET_INFO

print(f"PE types: {PE_TYPES}")
print(f"Seeds: {SEEDS}")
print(f"Datasets: {DATASETS}")

In [ ]:
# 5. List available checkpoints on Drive
# ===========================================================================
# >>> USER CONFIGURATION REQUIRED <<<
# Set CHECKPOINT_ROOT to the folder on YOUR Google Drive where you placed
# the 12 pretrained checkpoints. Expected layout:
#     <CHECKPOINT_ROOT>/learned_seed42/best_model.pth
#     <CHECKPOINT_ROOT>/learned_seed123/best_model.pth
#     ... (12 total)
# ===========================================================================
CHECKPOINT_ROOT = '/content/drive/MyDrive/Trained models_ImageNet100'  # <-- EDIT THIS

available = list_available_checkpoints(CHECKPOINT_ROOT)
print(f"Found {len(available)} checkpoints (expected: 12):")
for pe, seed in available:
    print(f"  {pe}_seed{seed}")

In [ ]:
# 6. Load one model (learned_seed42)  (no changes needed)
model = load_pretrained_model(
    CHECKPOINT_ROOT,
    pe_type='learned',
    seed=42,
    num_classes=100,
    device='cuda',
)
print(f"Model loaded: {type(model).__name__}")
print(f"PE type: {model.pe_type}")
print(f"Embed dim: {model.embed_dim}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

In [ ]:
# 7. Quick forward-pass sanity check  (no changes needed)
import torch
with torch.no_grad():
    dummy = torch.randn(2, 3, 224, 224).cuda()
    feats = model.forward_features(dummy)
    logits = model(dummy)
print(f"Features shape: {feats.shape}")  # should be [2, 768]
print(f"Logits shape: {logits.shape}")    # should be [2, 100]

In [ ]:
# 8. Load DTD (smallest dataset, fastest sanity check)  (no changes needed)
# DATA_ROOT is an ephemeral cache for the auto-downloaded dataset; it lives on
# the Colab instance and is wiped between sessions. Default location works.
DATA_ROOT = '/content/datasets'

train_loader = get_dataloader(
    'dtd', DATA_ROOT, split='train',
    batch_size=128, num_workers=2,
    augment=False, shuffle=False,
    download=True,
)
test_loader = get_dataloader(
    'dtd', DATA_ROOT, split='test',
    batch_size=128, num_workers=2,
    augment=False, shuffle=False,
    download=True,
)
print(f"DTD train size: {len(train_loader.dataset)}")
print(f"DTD test size: {len(test_loader.dataset)}")

In [ ]:
# 9. Extract features on full DTD  (no changes needed)
import numpy as np
from tqdm import tqdm
import time

@torch.no_grad()
def extract(loader):
    feats_list, labels_list = [], []
    for imgs, labels in tqdm(loader):
        imgs = imgs.cuda(non_blocking=True)
        feats = model.forward_features(imgs)
        feats_list.append(feats.cpu().numpy())
        labels_list.append(labels.numpy())
    return np.concatenate(feats_list), np.concatenate(labels_list)

t0 = time.time()
feats_train, labels_train = extract(train_loader)
feats_test, labels_test = extract(test_loader)
elapsed = time.time() - t0

print(f"Train features: {feats_train.shape}, labels: {labels_train.shape}")
print(f"Test features:  {feats_test.shape}, labels: {labels_test.shape}")
print(f"Elapsed: {elapsed:.1f}s")

In [ ]:
# 10. Quick kNN sanity baseline (k=20)  (no changes needed)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Normalize features (cosine similarity via L2 norm)
feats_train_norm = feats_train / np.linalg.norm(feats_train, axis=1, keepdims=True)
feats_test_norm = feats_test / np.linalg.norm(feats_test, axis=1, keepdims=True)

knn = KNeighborsClassifier(n_neighbors=20, metric='cosine', n_jobs=-1)
knn.fit(feats_train_norm, labels_train)
preds = knn.predict(feats_test_norm)
acc = accuracy_score(labels_test, preds)
print(f"kNN (k=20, cosine) accuracy on DTD: {acc:.4f}")
print(f"Random chance: {1 / DATASET_INFO['dtd']['num_classes']:.4f}")

## Interpretation of sanity result

If both the baseline accuracy (Cell 5) and the kNN result (Cell 10) are
well above chance, the pipeline is working. Move to `02_main_workflow.ipynb`
for the full sweep.